# Генерация ответов для тестовой выборки

Модель — encoder-decoder Transformer (57.6M параметров, 6 слоёв энкодера, 8 слоёв декодера), обученный с нуля
генерировать тело функции `solve()` по тексту задачи; полная программа собирается из него детерминированно
(`src/codefmt.py`). Описание данных, модели и обучения — в README репозитория.

Генерация (`src/infer.py`) на одной T4:
1. жадное декодирование для всех 1919 вопросов (fp16, KV-кэш, батчи отсортированы по длине вопроса);
2. пока позволяет лимит времени — 8 сэмплов на вопрос (температура 0.7); все кандидаты исполняются в песочнице,
   выбирается программа с самым частым числовым ответом, при равенстве — жадная (self-consistency).

Время считается с запуска скрипта, включая загрузку модели; сэмплирование прекращается так, чтобы уложиться
в 160 секунд. На Kaggle (GPU T4 x2) скрипт запускается с `CUDA_VISIBLE_DEVICES=0` и использует одну T4.

In [ ]:
import os

if not os.path.exists("hw2_nlp"):
    !git clone -q https://github.com/higemee/hw2_nlp.git
%cd hw2_nlp

Веса (`model.pt`, `tokenizer.json`): папка во входных данных Kaggle (`MODEL_DIR = None` — найти её
автоматически) или репозиторий на Hugging Face (`HF_REPO`).

In [ ]:
import glob

MODEL_DIR = None
HF_REPO = None

if HF_REPO:
    from huggingface_hub import snapshot_download
    MODEL_DIR = snapshot_download(HF_REPO, allow_patterns=["model.pt", "tokenizer.json"])
elif MODEL_DIR is None:
    MODEL_DIR = os.path.dirname(glob.glob("/kaggle/input/**/model.pt", recursive=True)[0])

from huggingface_hub import hf_hub_download

TEST = hf_hub_download("SlavaYus/tinysms_qa_12m", "data/test/test.parquet", repo_type="dataset")
print(MODEL_DIR, TEST)

In [ ]:
import time

start = time.time()
!CUDA_VISIBLE_DEVICES=0 python src/infer.py --model_dir {MODEL_DIR} --input {TEST} --out submission.jsonl
print(f"wall time: {time.time() - start:.1f}s")

In [ ]:
import json

with open("submission.jsonl") as f:
    submission = [json.loads(line) for line in f]
print(len(submission))
print(submission[0]["code"])